# APIM ❤️ FinOps

## FinOps Framework lab
![flow](../../images/finops-framework.gif)

This playground leverages the [FinOps Framework](https://www.finops.org/framework/) and Azure API Management to control AI costs. It uses the [token limit](https://learn.microsoft.com/en-us/azure/api-management/azure-openai-token-limit-policy) policy for each [product](https://learn.microsoft.com/en-us/azure/api-management/api-management-howto-add-products?tabs=azure-portal&pivots=interactive) and integrates [Azure Monitor alerts](https://learn.microsoft.com/en-us/azure/azure-monitor/alerts/alerts-overview) with [Logic Apps](https://learn.microsoft.com/en-us/azure/azure-monitor/alerts/alerts-logic-apps?tabs=send-email) to automatically disable APIM [subscriptions](https://learn.microsoft.com/en-us/azure/api-management/api-management-subscriptions) that exceed cost quotas.

### Result
![result](result.png)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the OpenAI model and version according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [9]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "westeurope"

aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]

models_config = [ { "name": "gpt-4.1-mini", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 200, "inputTokensMeterSku": "gpt 4.1 mini Inp glbl", "outputTokensMeterSku": "gpt 4.1 mini Outp glbl" }, 
                { "name": "gpt-4.1", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 200, "inputTokensMeterSku": "gpt 4.1 Inp glbl", "outputTokensMeterSku": "gpt 4.1 Outp glbl" },
                { "name": "DeepSeek-V3.2", "publisher": "DeepSeek",  "version": "1", "sku": "GlobalStandard", "capacity": 100, "inputTokensMeterSku": "V3.2 Inp glbl", "outputTokensMeterSku": "V3.2 Outp glbl"} ]

apim_sku = 'Basicv2'
apim_products_config = [{"name": "platinum", "displayName": "Platinum Product", "tpm": 2000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Monthly", "costQuota": 15 },
                    {"name": "gold", "displayName": "Gold Product", "tpm": 1000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Monthly", "costQuota": 10}, 
                    {"name": "silver", "displayName": "Silver Product", "tpm": 500, "tokenQuota": 1000000, "tokenQuotaPeriod": "Monthly", "costQuota": 5}]
apim_users_config = [ ]
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1", "product": "platinum" },
                    {"name": "subscription2", "displayName": "Subscription 2", "product": "gold" },
                    {"name": "subscription3", "displayName": "Subscription 3", "product": "silver" },
                     {"name": "subscription4", "displayName": "Subscription 4", "product": "silver" } ]

inference_api_path = "inference"  # path to the inference API in the APIM service
inference_api_type = "AzureOpenAI"  # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

currency_code = 'USD'

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 14:30:20.950777 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [10]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run("az ad signed-in-user show", "Retrieved az ad signed-in-user", "Failed to get az ad signed-in-user")
if output.success and output.json_data:
    current_user_object_id = output.json_data['id']

    

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:30:22.228294 :1s]
👉🏽 Current user: admin@MngEnvMCAP980490.onmicrosoft.com
👉🏽 Tenant ID: ba776ffb-8220-4d4c-b9c3-363e2ae9cde6
👉🏽 Subscription ID: 62910013-9757-43f8-a3b3-2e0143abd45b
⚙️ Running: az ad signed-in-user show 
✅ Retrieved az ad signed-in-user ⌚ 14:30:23.712119 :1s]


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations. 

⚠️ Retry this step if you get deployment error: `workspace not active` 

In [11]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimUsersConfig": { "value": apim_users_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "apimProductsConfig": { "value": apim_products_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name lab-finops-framework 
👉🏽 Using existing resource group 'lab-finops-framework'
⚙️ Running: az deployment group create --name finops-framework --resource-group lab-finops-framework --template-file main.bicep --parameters params.json 
✅ Deployment 'finops-framework' succeeded ⌚ 14:33:20.331460 :54s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the required outputs from the Bicep deployment.

In [12]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    pricing_dcr_endpoint = utils.get_deployment_output(output, 'pricingDCREndpoint', 'Pricing DCR Endpoint')
    pricing_dcr_immutable_id = utils.get_deployment_output(output, 'pricingDCRImmutableId', 'Pricing DCR ImmutableId')
    pricing_dcr_stream = utils.get_deployment_output(output, 'pricingDCRStream', 'Pricing DCR Stream')
    subscription_quota_dcr_endpoint = utils.get_deployment_output(output, 'subscriptionQuotaDCREndpoint', 'Subscription Quota DCR Endpoint')
    subscription_quota_dcr_immutable_id = utils.get_deployment_output(output, 'subscriptionQuotaDCRImmutableId', 'Subscription Quota DCR ImmutableId')
    subscription_quota_dcr_stream = utils.get_deployment_output(output, 'subscriptionQuotaDCRStream', 'Subscription Quota DCR Stream')
    
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")


⚙️ Running: az deployment group show --name finops-framework -g lab-finops-framework 
✅ Retrieved deployment: finops-framework ⌚ 14:33:22.869431 :2s]
👉🏽 APIM API Gateway URL: https://apim-ycvzj4i7bf6vy.azure-api.net
👉🏽 Pricing DCR Endpoint: https://dcr-pricing-ycvzj4i7bf6vy-b5e1-westeurope.logs.z1.ingest.monitor.azure.com
👉🏽 Pricing DCR ImmutableId: dcr-c29da52624be4898942a82eb4d6889b5
👉🏽 Pricing DCR Stream: Custom-Json-PRICING_CL
👉🏽 Subscription Quota DCR Endpoint: https://dcr-quota-ycvzj4i7bf6vy-bwpc-westeurope.logs.z1.ingest.monitor.azure.com
👉🏽 Subscription Quota DCR ImmutableId: dcr-153b68cb93b04d26a42d58a1946df718
👉🏽 Subscription Quota DCR Stream: Custom-Json-SUBSCRIPTION_QUOTA_CL
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****9dd8
👉🏽 Subscription Name: subscription2
👉🏽 Subscription Key: ****6748
👉🏽 Subscription Name: subscription3
👉🏽 Subscription Key: ****6693
👉🏽 Subscription Name: subscription4
👉🏽 Subscription Key: ****4bbb


<a id='pricing'></a>
### 🔍 Display retail pricing info based on the [pricing API](https://learn.microsoft.com/en-us/rest/api/cost-management/retail-prices/azure-retail-prices)



In [13]:
import requests
from tabulate import tabulate 

def build_pricing_table(json_data, table_data):
    for item in json_data['Items']:
        meter = item['meterName']
        table_data.append([item['armRegionName'], item['armSkuName'], item['retailPrice']*1000])

table_data = []
table_data.append(['Region', 'SKU', 'Retail Price'])
for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']    
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&$filter=serviceName eq 'Foundry Models' and unitOfMeasure eq '1K' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        build_pricing_table(prices_json, table_data)
    print(tabulate(table_data, headers='firstrow', tablefmt='psql'))


+---------------+---------------------------------------------+----------------+
| Region        | SKU                                         |   Retail Price |
|---------------+---------------------------------------------+----------------|
| swedencentral | o3 mini 0131 Batch Outp Data Zone           |          2.42  |
| swedencentral | gpt 4.1 nano cached Inp glbl                |          0.025 |
| swedencentral | gpt4omini-rt-aud1217 Outp regnl             |         24.2   |
| swedencentral | Model 5 Inp glbl                            |          0.033 |
| swedencentral | gpt-4o-aud-0603-txt Inp DZone               |          2.75  |
| swedencentral | gpt-4o-rt-aud-0603 Outp glbl                |         80     |
| swedencentral | Phi-4-reasoning-Output                      |          0.5   |
| swedencentral | o1-pro Inp regnl                            |        181.5   |
| swedencentral | o3 0416 Batch Outp glbl                     |          4     |
| swedencentral | gpt 4.1 In

<a id='4'></a>
### 4️⃣ Load the pricing data into Azure Monitor custom table

👉 This script uses retail price information. Please adjust it to apply a discount or to use a flat rate with PTUs.   
👉 We are multiplying by 1000 to get the retail price per 1K tokens.   
👉 Deploy this script as a [job](https://learn.microsoft.com/en-us/azure/container-apps/jobs?tabs=azure-cli) to run automatically on a predefined schedule.

In [14]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=pricing_dcr_endpoint, credential=credential, logging_enable=False)

for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&$filter=serviceName eq 'Foundry Models' and unitOfMeasure eq '1K' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        if prices_json and 'Items' in prices_json:
            for deployment in models_config:
                input_tokens_price = next((item['retailPrice'] * 1000 for item in prices_json['Items'] if item.get('skuName') == deployment.get("inputTokensMeterSku")), None)
                output_tokens_price = next((item['retailPrice'] * 1000 for item in prices_json['Items'] if item.get('skuName') == deployment.get("outputTokensMeterSku")), None)
                utils.print_info(f"Adding model {deployment.get("name")} with input / output tokens price {input_tokens_price} / {output_tokens_price}")
                body = [{ "TimeGenerated": str(datetime.now(timezone.utc)),
                        "Model": deployment.get("name"),
                        "InputTokensPrice": input_tokens_price,
                        "OutputTokensPrice": output_tokens_price }]
                try:
                    client.upload(rule_id=pricing_dcr_immutable_id, stream_name=pricing_dcr_stream, logs=body)
                    utils.print_ok(f"Upload succeeded for model {deployment.get("name")}")
                except HttpResponseError as e:
                    utils.print_error(f"Upload failed: {e}")            


👉🏽 Adding model gpt-4.1-mini with input / output tokens price 0.4 / 1.6
✅ Upload succeeded for model gpt-4.1-mini ⌚ 14:33:27.135800 
👉🏽 Adding model gpt-4.1 with input / output tokens price 2.0 / 8.0
✅ Upload succeeded for model gpt-4.1 ⌚ 14:33:27.339531 
👉🏽 Adding model DeepSeek-V3.2 with input / output tokens price 0.58 / 1.6800000000000002
✅ Upload succeeded for model DeepSeek-V3.2 ⌚ 14:33:27.541814 


<a id='5'></a>
### 5️⃣ Load the Subscription Quota into Azure Monitor custom table


In [18]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=subscription_quota_dcr_endpoint, credential=credential, logging_enable=False)

for subscription in apim_subscriptions_config:
    for product in apim_products_config:
        if product.get("name") == subscription.get("product"):
            cost_quota = product.get("costQuota")
            utils.print_info(f"Adding {subscription.get('name')} with cost quota {cost_quota}")
            body = [{ 
                "TimeGenerated": str(datetime.now(timezone.utc)),
                "Subscription": subscription.get("name"),
                "Email": subscription.get("email"),
                "CostQuota": cost_quota
            }]
            try:
                client.upload(rule_id=subscription_quota_dcr_immutable_id, stream_name=subscription_quota_dcr_stream, logs=body)
                utils.print_ok(f"Upload succeeded for {subscription.get("name")}")
            except HttpResponseError as e:
                utils.print_error(f"Upload failed: {e}")            


👉🏽 Adding subscription1 with cost quota 15
✅ Upload succeeded for subscription1 ⌚ 15:10:45.708609 
👉🏽 Adding subscription2 with cost quota 10
✅ Upload succeeded for subscription2 ⌚ 15:10:46.086674 
👉🏽 Adding subscription3 with cost quota 5
✅ Upload succeeded for subscription3 ⌚ 15:10:46.343368 
👉🏽 Adding subscription4 with cost quota 5
✅ Upload succeeded for subscription4 ⌚ 15:10:46.505248 


<a id='sdk'></a>
### 🧪 Execute multiple runs using the Azure OpenAI Python SDK

👉 We will send requests with random subscription and models. Adjust the `sleep_time_ms` and the number of `runs` to your test scenario.


In [19]:
import time, random
from openai import AzureOpenAI

runs = 10
sleep_time_ms = 100

for i in range(runs):
    apim_subscription = random.choice(apim_subscriptions)
    openai_model = random.choice(models_config)
    client = AzureOpenAI(
        azure_endpoint = f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key = apim_subscription.get("key"),
        api_version = inference_api_version
    )
    try:
        response = client.chat.completions.create(
            model = str(openai_model.get('name')),
            messages = [
                {"role": "user", "content": "Can you tell me the time, please?"}
            ],
            extra_headers = {"x-user-id": "alex"}
        )
        print(f"▶️ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] 💬 {response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] Error: {e}")
    time.sleep(sleep_time_ms/1000)


▶️ Run 1/10: [subscription3 w/ DeepSeek-V3.2] 💬 I can't provide the current time as I don't have real-time access to that information. You can check the time on your device, ask a voice assistant, or look at a clock nearby.
▶️ Run 2/10: [subscription3 w/ DeepSeek-V3.2] 💬 I can’t provide the current time as I don’t have real-time access.  
You can check the time on your device, or ask a voice assistant like Siri, Google Assistant, or Alexa. 😊
▶️ Run 3/10: [subscription1 w/ gpt-4.1-mini] 💬 I’m not able to provide the current time. You might want to check a clock, your phone, or a computer for the most accurate time.
▶️ Run 4/10: [subscription3 w/ DeepSeek-V3.2] 💬 I don’t have access to real-time data, so I can’t provide the current time. You can check the time on your device, ask a voice assistant, or look at a nearby clock.
▶️ Run 5/10: [subscription1 w/ gpt-4.1] 💬 I'm sorry, but I can't access real-time information such as the current time. Please check a clock, your device, or another

<a id='workbooks'></a>
### 🔍 Open the dashboard and workbooks in the Azure Portal

👉 The Cost Analysis workbook contains information on the total costs and quotas for each subscription.  
👉 The [Azure OpenAI Insights workbook](https://github.com/dolevshor/Azure-OpenAI-Insights) provides comprehensive details about service and model usage. Credits to [Dolev Shor](https://github.com/dolevshor/Azure-OpenAI-Insights).  
👉 The [Alerts workbook](https://github.com/microsoft/AzureMonitorCommunity/tree/master/Azure%20Services) provides information about the alerts triggered by Azure Monitor.  

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.